In [1]:
import pandas as pd
from sklearn import model_selection
from pipeline import (
    create_attributes_pipeline,
    create_encoder_pipeline,
    model_attributes,
)


def load_dataset(name_dataset):
    #leemos el dataset de futbol uruguayo
    df = pd.read_csv(name_dataset)

    #nos quedamos con las columnas que nos interesan para el clasificador
    df = df.drop(columns=["full_time", "competition", "home_ident", "away_ident", "home_country", "away_country", "home_code", "away_code", "home_continent", "away_continent", "continent", "level"])

    #convertimos las columnas a los tipos de datos correctos
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    df["gh"] = pd.to_numeric(df["gh"], errors="raise").astype(int)
    df["ga"] = pd.to_numeric(df["ga"], errors="raise").astype(int)

    #ordenamos el dataset por fecha y por equipos, y eliminamos duplicados
    df = (
        df.drop_duplicates()
        .sort_values(["date", "home", "away"], kind="stable")
        .reset_index(drop=True)
    )

    #creamos la columna result, que es el resultado del partido, L si gana el local, V si gana el visitante y E si empatan
    df["result"] = df.apply(
        lambda row: "L" if row["gh"] > row["ga"] else ("V" if row["gh"] < row["ga"] else "E"), axis=1
    )

    return df

In [2]:
dataset = load_dataset("futbol_uruguayo.csv")

#creamos los atributos historicos sobre el dataset completo
#cada partido utiliza unicamente los resultados anteriores a su fecha
attributes_pipeline = create_attributes_pipeline(
    years_limit=10,
    matches_limit=5,
)

dataset_with_attributes = attributes_pipeline.fit_transform(
    dataset
)

In [3]:
print(dataset_with_attributes.head())

                   home                              away       date  gh  ga  \
0           Bella Vista                 Defensor Sporting 1932-03-05   1   2   
1            CA Penarol                       River Plate 1932-03-05   1   1   
2       Central Espanol        Rampla Juniors Futbol Club 1932-03-05   1   0   
3  Montevideo Wanderers                       Racing Club 1932-03-05   3   0   
4              Nacional  Institucion Atletica Sud America 1932-03-05   2   0   

  result record last_matches goal_difference  local_experience  \
0      V      E            E               E                 0   
1      E      E            E               E                 0   
2      L      E            E               E                 0   
3      L      E            E               E                 0   
4      L      E            E               E                 0   

   away_experience  record_enough  
0                0              0  
1                0              0  
2             

In [4]:
#separamos cronologicamente el conjunto de entrenamiento y el de evaluacion
#la evaluacion se mantiene separada hasta haber elegido los hiperparametros

train = dataset_with_attributes[
    dataset_with_attributes["date"] < pd.Timestamp("2024-01-01")
].copy()

test = dataset_with_attributes[
    (dataset_with_attributes["date"] >= pd.Timestamp("2024-01-01"))
    & (dataset_with_attributes["date"] < pd.Timestamp("2026-01-01"))
].copy()

print("Cantidad de partidos de entrenamiento:", len(train))
print("Cantidad de partidos de evaluacion:", len(test))

Cantidad de partidos de entrenamiento: 14734
Cantidad de partidos de evaluacion: 472


In [ ]:
#separamos los atributos de entrada y la clase que queremos predecir
X_train = train[model_attributes].copy()
y_train = train["result"].copy()

X_test = test[model_attributes].copy()
y_test = test["result"].copy()

#el encoder aprende la codificacion usando solamente entrenamiento
encoder_pipeline = create_encoder_pipeline()

X_train_encoded = encoder_pipeline.fit_transform(
    X_train
)

#en evaluacion reutilizamos exactamente la codificacion aprendida antes
X_test_encoded = encoder_pipeline.transform(
    X_test
)

print(X_train_encoded.head(20))

    record  last_matches  goal_difference  local_experience  away_experience  \
0        1             1                1                 0                0   
1        1             1                1                 0                0   
2        1             1                1                 0                0   
3        1             1                1                 0                0   
4        1             1                1                 0                0   
5        1             2                2                 1                1   
6        2             2                2                 1                1   
7        1             1                2                 1                1   
8        2             2                2                 1                1   
9        1             1                1                 1                1   
10       1             1                2                 1                1   
11       2             2                

In [ ]:
import sys
import os

# Agrega la carpeta padre (Tarea1) al path de búsqueda de Python
sys.path.append(os.path.abspath(".."))


#hacemos el arbol de decision usando solamente el conjunto de entrenamiento
from decisionTree.clasifier import Clasifier as DecisionTreeClassifier

modelo = DecisionTreeClassifier(0.003)

modelo.fit(X_train_encoded, y_train)
modelo.tree.print_tree()